# L1: Automated Project: Planning, Estimation, and Allocation

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

## Initial Imports

In [39]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

import os
os.environ["LITELLM_ENDPOINT"] = "v1" # O forzar el comportamiento de la API estándar 
import yaml
from dotenv import load_dotenv
from crewai import Agent, Task, Crew

# Load environment variables
load_dotenv()

# Asegúrate de que la API key esté disponible para LiteLLM
# (LiteLLM busca automáticamente la variable GOOGLE_API_KEY en el entorno)
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Configurar el LLM de Gemini usando el formato compatible con LiteLLM/CrewAI
# Reemplazamos gemini-1.5-flash por el identificador estándar actual (ej. gemini/gemini-2.5-flash)
# Configurar el LLM de Gemini con uno de los modelos disponibles en tu lista
llm_gemini = "gemini/gemini-3.5-flash"

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix - Tips and Help"</em> Lesson.</p>

## Set OpenAI Model

In [ ]:
#os.environ['OPENAI_MODEL_NAME'] = 'gemini/gemini-1.5-flash'

## Loading Tasks and Agents YAML files

In [40]:
# Define file paths for YAML configurations
files = {
    'agents': '../config/agents.yaml',
    'tasks': '../config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Create Pydantic Models for Structured Output

In [41]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of tasks with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones")

## Create Crew, Agents and Tasks

In [42]:
# Configurar el LLM de Gemini correctamente para CrewAI
from crewai import Agent, Task, Crew
from langchain_google_genai import ChatGoogleGenerativeAI

# O bien usa directamente la nomenclatura compatible con LiteLLM que acepta CrewAI:
llm_gemini = "gemini/gemini-3.5-flash" # O la versión estable actual de tu entorno
# Creando los agentes usando el string del modelo compatible con LiteLLM
project_planning_agent = Agent(
    config=agents_config['project_planning_agent'],
    llm=llm_gemini,
    verbose=True
)

estimation_agent = Agent(
    config=agents_config['estimation_agent'],
    llm=llm_gemini,
    verbose=True
)

resource_allocation_agent = Agent(
    config=agents_config['resource_allocation_agent'],
    llm=llm_gemini,
    verbose=True
)

# Creating Tasks
task_breakdown = Task(
    config=tasks_config['task_breakdown'],
    agent=project_planning_agent
)

time_resource_estimation = Task(
    config=tasks_config['time_resource_estimation'],
    agent=estimation_agent
)

resource_allocation = Task(
    config=tasks_config['resource_allocation'],
    agent=resource_allocation_agent,
    output_json=ProjectPlan  # Aquí vinculamos el modelo Pydantic que creamos antes para estructurar la salida
)

# Creating Crew
crew = Crew(
  agents=[
    project_planning_agent,
    estimation_agent,
    resource_allocation_agent
  ],
  tasks=[
    task_breakdown,
    time_resource_estimation,
    resource_allocation
  ],
  verbose=True
)

## Crew's Inputs

In [43]:
from IPython.display import display, Markdown

project = 'Website'
industry = 'Technology'
project_objectives = 'Create a website for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

# Format the dictionary as Markdown for a better display in Jupyter Lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}
**Project Requirements:**
{project_requirements}
"""
# Display the formatted output as Markdown
display(Markdown(formatted_output))


**Project Type:** Website

**Project Objectives:** Create a website for a small business

**Industry:** Technology

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)

**Project Requirements:**

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust



## Kicking off the crew

In [44]:
# El diccionario con los datos de entrada
inputs = {
  'project_type': project,
  'project_objectives': project_objectives,
  'industry': industry,
  'team_members': team_members,
  'project_requirements': project_requirements
}

# Ejecutar la tripulación de forma asincrónica en el Notebook
result = await crew.kickoff_async(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 07a06921-8fc5-44bb-bf40-61bdc3306828                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  ID: b0b7defe-03c8-414a-bfe2-94b18f969475                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Task: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # PROJECT IMPLEMENTATION PLAN: SMALL BUSINESS WEBSITE PROJECT                                                  │
│                                                                                                                 │
│  **Project Name:** Small Business Modern Web Presence                                                           │
│  **Project Lead:** John Doe (Project Manager)                                                                   │
│  **Total Estimated Duration:** 38 Business Days                                                                 │
│  **Target Completion Window:** Day 1 to Day 38                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Summary & Team Resource Allocation                                                             │
│                                                                                                                 │
│  As the Project Manager, I have structured this project to ensure zero friction between design, development,    │
│  optimization, and quality assurance. Each task aligns directly with the core business requirements, ensuring   │
│  a performant, mobile-first, conversion-optimized, and SEO-friendly website for the business.                   │
│                                                                                                                 │
│  ### Team Roles & Assigned Resources:                                                                           │
│  * **John Doe (Project Manager):** Project governance, requirements alignment, stakeholder management,          │
│  deployment oversight.                                                                                          │
│  * **Bob Smith (UI/UX Designer):** Wireframing, interface design, responsive layouts, design assets, UI         │
│  component library.                                                                                             │
│  * **Jane Doe (Software Engineer):** Frontend & backend architecture, responsive coding, API integrations       │
│  (Maps, Forms), SEO technical setup, site speed optimization.                                                   │
│  * **Alice Johnson (QA Engineer):** Functional testing, cross-browser compatibility, responsive/mobile          │
│  testing, form & link validation.                                                                               │
│  * **Tom Brown (QA Engineer):** Performance benchmarking, Lighthouse audits, technical SEO validation,          │
│  security & SSL audit.                                                                                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Detailed Work Breakdown Structure (WBS) & Task S

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (7 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resource Allocation Strategist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  tasks=[TaskEstimate(task_name='Task 1.1: Project Kickoff & Technical Architecture Definition',                 │
│  estimated_time_hours=16.67, required_resources=['John Doe', 'Jane Doe', 'Bob Smith']),                         │
│  TaskEstimate(task_name='Task 1.2: Information Architecture & Navigation System Mapping',                       │
│  estimated_time_hours=16.33, required_resources=['Bob Smith', 'Jane Doe']), TaskEstimate(task_name='Task 2.1:   │
│  High-Fidelity Responsive Design & UI Kit Creation', estimated_time_hours=49.0, required_resources=['Bob        │
│  Smith', 'Jane Doe']), TaskEstimate(task_name='Task 3.1: Core Responsive Framework & Navigation                 │
│  Implementation', estimated_time_hours=25.0, required_resources=['Jane Doe', 'Bob Smith']),                     │
│  TaskEstimate(task_name='Task 3.2: Page Development – Home & Testimonials Module', estimated_time_hours=25.0,   │
│  required_resources=['Jane Doe']), TaskEstimate(task_name='Task 3.3: Page Development – About Us & Services     │
│  Pages', estimated_time_hours=25.0, required_resources=['Jane Doe']), TaskEstimate(task_name='Task 3.4: Page    │
│  Development – Contact Us Page (Form & Map Integration)', estimated_time_hours=25.0, required_resources=['Jane  │
│  Doe']), TaskEstimate(task_name='Task 3.5: Page Development – Blog Section & Social Sharing Integration',       │
│  estimated_time_hours=33.33, required_resources=['Jane Doe']), TaskEstimate(task_name='Task 4.1: Fast Loading   │
│  Speed & Technical SEO Implementation', estimated_time_hours=25.0, required_resources=['Jane Doe', 'Tom         │
│  Brown']), TaskEstimate(task_name='Task 5.1: Cross-Browser & Responsive Design Testing',                        │
│  estimated_time_hours=32.67, required_resources=['Alice Johnson', 'Jane Doe']), TaskEstimate(task_name='Task    │
│  5.2: Performance, Accessibility & SEO Audit', estimated_time_hours=32.67, required_resources=['Tom Brown',     │
│  'Jane Doe']), TaskEstimate(task_name='Task 5.3: Bug Remediation & Polish', estimated_time_hours=33.33,         │
│  required_resources=['Jane Doe', 'Bob Smith', 'Alice Johnson', 'Tom Brown']), TaskEstimate(task_name='Task      │
│  6.1: Production Deployment & Go-Live', estimated_time_hours=20.33, required_resources=['John Doe', 'Jane       │
│  Doe'])] milestones=[Milestone(milestone_name='Phase 1: Initiation, Requirements & Architecture',               │
│  tasks=['1.1', '1.2']), Milestone(milestone_name='Phase 2: UI/UX Design & Prototyping', tasks=['2.1']),         │
│  Milestone(milestone_name='Phase 3: Frontend & Backend Development', tasks=['3.1', '3.2', '3.3', '3.4',         │
│  '3.5']), Milestone(milestone_name='Phase 4: Optimization & Technical SEO', tasks=['4.1']),                     │
│  Milestone(milestone_name='Phase 5: Quality Assurance & Defect Remediation', tasks=['5.1', '5.2', '5.3']),      │
│  Milestone(milestone_name='Phase 6: Production Launch & Handover', tasks=['6.1'])]                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Strategically allocate tasks for the Website project to team members based on their skills,              │
│  availability, and current workload. Ensure that each task is assigned to the most suitable team member and     │
│  that the workload is evenly distributed.                                                                       │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  Agent: Resource Allocation Strategist                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 07a06921-8fc5-44bb-bf40-61bdc3306828                                                                       │
│  Final Output: {"tasks":[{"task_name":"Task 1.1: Project Kickoff & Technical Architecture                       │
│  Definition","estimated_time_hours":16.67,"required_resources":["John Doe","Jane Doe","Bob                      │
│  Smith"]},{"task_name":"Task 1.2: Information Architecture & Navigation System                                  │
│  Mapping","estimated_time_hours":16.33,"required_resources":["Bob Smith","Jane Doe"]},{"task_name":"Task 2.1:   │
│  High-Fidelity Responsive Design & UI Kit Creation","estimated_time_hours":49.0,"required_resources":["Bob      │
│  Smith","Jane Doe"]},{"task_name":"Task 3.1: Core Responsive Framework & Navigation                             │
│  Implementation","estimated_time_hours":25.0,"required_resources":["Jane Doe","Bob Smith"]},{"task_name":"Task  │
│  3.2: Page Development – Home & Testimonials Module","estimated_time_hours":25.0,"required_resources":["Jane    │
│  Doe"]},{"task_name":"Task 3.3: Page Development – About Us & Services                                          │
│  Pages","estimated_time_hours":25.0,"required_resources":["Jane Doe"]},{"task_name":"Task 3.4: Page             │
│  Development – Contact Us Page (Form & Map                                                                      │
│  Integration)","estimated_time_hours":25.0,"required_resources":["Jane Doe"]},{"task_name":"Task 3.5: Page      │
│  Development – Blog Section & Social Sharing                                                                    │
│  Integration","estimated_time_hours":33.33,"required_resources":["Jane Doe"]},{"task_name":"Task 4.1: Fast      │
│  Loading Speed & Technical SEO Implementation","estimated_time_hours":25.0,"required_resources":["Jane          │
│  Doe","Tom Brown"]},{"task_name":"Task 5.1: Cross-Browser & Responsive Design                                   │
│  Testing","estimated_time_hours":32.67,"required_resources":["Alice Johnson","Jane Doe"]},{"task_name":"Task    │
│  5.2: Performance, Accessibility & SEO Audit","estimated_time_hours":32.67,"required_resources":["Tom           │
│  Brown","Jane Doe"]},{"task_name":"Task 5.3: Bug Remediation &                                                  │
│  Polish","estimated_time_hours":33.33,"required_resources":["Jane Doe","Bob Smith","Alice Johnson","Tom         │
│  Brown"]},{"task_name":"Task 6.1: Production Deployment &                                                       │
│  Go-Live","estimated_time_hours":20.33,"required_resources":["John Doe","Jane                                   │
│  Doe"]}],"milestones":[{"milestone_name":"Phase 1: Initiation, Requirements &                                   │
│  Architecture","tasks":["1.1","1.2"]},{"milestone_name":"Phase 2: UI/UX Design &                                │
│  Prototyping","tasks":["2.1"]},{"milestone_name":"Phase 3: Frontend & Backend                                   │
│  Development","tasks":["3.1","3.2","3.3","3.4","3.5"]},{"milestone_name":"Phase 4: Optimization & Technical     │
│  SEO","tasks":["4.1"]},{"milestone_name":"Phase 5: Quality Assurance & Defect                                   │
│  Remediation","tasks":["5.1","5.2","5.3"]},{"milestone_name":"Phase 6: Production Launch &                      │
│  Handover","tasks":["6.1"]}]}                                                                                   │
│                                                       

## Usage Metrics and Costs

Let’s see how much it would cost each time if this crew runs at scale.

In [ ]:
import pandas as pd

costs = 0.150 * (crew.usage_metrics.prompt_tokens + crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([crew.usage_metrics.dict()])
df_usage_metrics

## Result

In [ ]:
result.pydantic.dict()

## Inspect further

In [ ]:
tasks = result.pydantic.dict()['tasks']
df_tasks = pd.DataFrame(tasks)

# Display the DataFrame as an HTML table
df_tasks.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

### Inspecting Milestones

In [ ]:
milestones = result.pydantic.dict()['milestones']
df_milestones = pd.DataFrame(milestones)

# Display the DataFrame as an HTML table
df_milestones.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)